# Document Classification: Naive Bayes vs KNN
## Using AG News Dataset

### Learning Objectives
- Understand how to convert text data into numerical vectors
- Implement document classification using Naive Bayes and KNN algorithms
- Compare performance and characteristics of both algorithms

### Lab Instructions
- Execute code cells in order
- Complete sections marked with `TODO`

### Dataset Information
**AG News Dataset**
- One of the most widely used benchmark datasets for text classification
- Contains news articles from over 2000 news sources
- 4 categories: World, Sports, Business, Science/Technology
- 120,000 training samples, 7,600 test samples

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

---
## 2. Theoretical Background

### 2.1 Naive Bayes Classifier

**Core Principle**
- A probabilistic classifier based on Bayes' theorem
- Assumes all features (words) are independent of each other ("Naive" assumption)

**Mathematical Formula**
$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$

Where:
- $P(C|X)$: Probability of category C given document X
- $P(X|C)$: Likelihood of document X in category C
- $P(C)$: Prior probability of category C

**Advantages**
- Very fast training speed
- Effective even with small datasets
- Performs well in high-dimensional spaces
- Provides probability scores for interpretability

**Disadvantages**
- Independence assumption is unrealistic
- Cannot capture relationships between words

### 2.2 K-Nearest Neighbors (KNN)

**Core Principle**
- A distance-based classification algorithm
- Classifies new data by majority vote of K nearest neighbors

**Distance Metric**
- Cosine similarity is commonly used for text data
$$\text{similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

**Advantages**
- Intuitive and easy to understand
- No training phase (Lazy Learning)
- Can model non-linear decision boundaries

**Disadvantages**
- Slow prediction time (computes distance to all training data)
- High memory usage
- Performance degrades in high dimensions (curse of dimensionality)
- Sensitive to the choice of K

---
## 3. Load AG News Dataset

In [ ]:
# Install datasets library if needed
# !pip install datasets

In [ ]:
from datasets import load_dataset

# Load AG News dataset
print("Loading AG News dataset...")
dataset = load_dataset('ag_news')

print(f"Dataset loaded successfully")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

Loading AG News dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset loaded successfully
Train size: 120000
Test size: 7600


In [ ]:
# TODO: Convert the dataset to pandas DataFrame
# Hint: Use pd.DataFrame(dataset['train']) and pd.DataFrame(dataset['test'])

# train_df =
# test_df =

# Map label numbers to category names
label_names = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
# train_df['category'] =
# test_df['category'] =

In [ ]:
# For practical training time, we'll use a subset
TRAIN_SIZE = 10000
TEST_SIZE = 2000

train_sample = train_df.sample(n=TRAIN_SIZE, random_state=42)
test_sample = test_df.sample(n=TEST_SIZE, random_state=42)

print(f"Using {TRAIN_SIZE} training samples and {TEST_SIZE} test samples")

In [ ]:
# TODO: Display the category distribution in the training set
# Hint: Use value_counts()


---
## 4. Exploratory Data Analysis

In [ ]:
# Analyze text length
train_sample['word_count'] = train_sample['text'].str.split().str.len()

print("Text length statistics:")
print(train_sample['word_count'].describe())

In [ ]:
# TODO: Create a bar plot showing the category distribution
# Hint: Use value_counts().plot(kind='bar')


---
## 5. Prepare Train and Test Sets

In [ ]:
# Extract features and labels
X_train = train_sample['text']
y_train = train_sample['category']
X_test = test_sample['text']
y_test = test_sample['category']

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

---
## 6. Text Vectorization

### TF-IDF (Term Frequency-Inverse Document Frequency)
- **TF**: Frequency of a term in a document
- **IDF**: Rarity of a term across all documents
- **Result**: Higher weights for words that characterize specific documents

In [ ]:
# TF-IDF Vectorizer
vectorizer_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    stop_words='english'
)

print("Vectorizing text data...")
start = time()
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)
vectorize_time = time() - start

print(f"Vectorization completed in {vectorize_time:.2f} seconds")
print(f"Vectorized shape: {X_train_tfidf.shape}")

---
## 7. Naive Bayes Model

In [ ]:
# TODO: Create and train a Naive Bayes model
# Hint: Use MultinomialNB(alpha=0.1)

# nb_model =
# nb_model.fit(?, ?)
# y_pred_nb = nb_model.predict(?)

In [ ]:
# TODO: Calculate and print the accuracy
# Hint: Use accuracy_score(y_test, y_pred_nb)

# acc_nb =
# print(f"Naive Bayes Accuracy: {acc_nb:.4f}")

---
## 8. KNN Model

In [ ]:
# KNN with k=5
print("Training KNN with k=5...")
knn_model = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)

start = time()
knn_model.fit(X_train_tfidf, y_train)
train_time_knn = time() - start

start = time()
y_pred_knn = knn_model.predict(X_test_tfidf)
predict_time_knn = time() - start

acc_knn = accuracy_score(y_test, y_pred_knn)

print(f"Training time: {train_time_knn:.4f}s")
print(f"Prediction time: {predict_time_knn:.4f}s")
print(f"Accuracy: {acc_knn:.4f}")

In [ ]:
# TODO: Try KNN with k=3 and k=7
# Compare which K value gives the best accuracy


---
## 9. Model Comparison

In [ ]:
# TODO: Create a bar plot comparing Naive Bayes and KNN accuracy


---
## 10. Performance Analysis

In [ ]:
# Classification report
print("Classification Report (Naive Bayes):")
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_nb)
categories = sorted(train_sample['category'].unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories)
plt.title('Confusion Matrix (Naive Bayes)', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

### Question: Which category pairs are most often confused?

---
## 11. Cross-Validation

In [ ]:
# TODO: Perform 5-fold cross-validation for Naive Bayes
# Hint: Use cross_val_score()

# X_all = vectorizer_tfidf.fit_transform(train_sample['text'])
# y_all = train_sample['category']
# cv_scores = cross_val_score(nb_model, X_all, y_all, cv=5)
# print(f"CV Mean: {cv_scores.mean():.4f}")

---
## 12. Predict New Documents

In [ ]:
# TODO: Write your own news article and predict its category

my_article = [
    "Apple announces new iPhone with advanced AI capabilities and improved camera system.",
    "The stock market reached new highs today as investors showed confidence in economic recovery.",
    "Scientists discover a new planet in the habitable zone of a nearby star system.",
    "The national team won the championship after an exciting final match against their rivals."
]  # Write your article here

# my_article_vec = vectorizer_tfidf.transform([my_article])
# prediction = nb_model.predict(my_article_vec)
# print(f"Predicted category: {prediction[0]}")